In [ ]:
!pip install transformers accelerate sacremoses gdown

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random
from collections import Counter
from tqdm.auto import tqdm

In [ ]:
# import gdown
# import zipfile
# import os
# import torch
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# MODEL_INFO = {
#     "mbart_ft": {"id": "1O6G2SH9ri1nb6w4YR_263uJ_0p8do6T2", "archive": "mbart.zip", "folder": "mbart_fine_tuned_essential2"},
#     "nllb_ft": {"id": "1uEySNZMYAylShxVxrDuhjlKavXlx_9P8", "archive": "nllb.zip", "folder": "fine_tuned_nllb_tourism_essential2"},
#     "helsinki_ft": {"id": "1rPNS5q2BeurL6yJIV1YZobR_QlayQtbj", "archive": "helsinki.zip", "folder": "fine_tuned_helsinki_tourism2"},

#     "custom_pt": {"id": "1_KkZH_E4xKg7n_DY56lP13qJPj9roteg", "filename": "LSTM2.pt"},
# }

# # Define a directory for all downloads
# DOWNLOAD_DIR = "./downloaded_models"
# os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [ ]:
# loaded_hf_models = {}

# print("--- Downloading and Extracting Hugging Face Models ---")
# for key, info in MODEL_INFO.items():
#     if 'archive' in info:
#         archive_path = os.path.join(DOWNLOAD_DIR, info['archive'])

#         # Download
#         print(f"Downloading {key}...")
#         gdown.download(id=info['id'], output=archive_path, quiet=True)

#         # Extract
#         with zipfile.ZipFile(archive_path, 'r') as zip_ref:
#             # Extract to the download directory
#             zip_ref.extractall(DOWNLOAD_DIR)

#         # Load Model and Tokenizer
#         model_path = os.path.join(DOWNLOAD_DIR, info['folder'])
#         print(f"Loading model from {model_path}...")

#         loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
#         loaded_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

#         loaded_hf_models[key] = {'model': loaded_model, 'tokenizer': loaded_tokenizer}
#         print(f"Loaded {key.upper()} successfully.")

# # Access your models, e.g., loaded_hf_models['mbart_ft']['model']

In [ ]:
test_df = pd.read_csv("https://drive.google.com/uc?id=1eX6RCAxYS8g_DAOgg4pvIEqzph6Y15kU")
random_rows = test_df['text'].sample(5)

# LSTM

## Declare

In [ ]:
class Vocabulary:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = 1

    def __len__(self):
        return len(self.stoi)

    @staticmethod
    def tokenizer(text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

In [ ]:
class ParallelDataset(Dataset):
    def __init__(self, df, src_col, trg_col, src_vocab, trg_vocab):
        self.df = df
        self.src_col = src_col
        self.trg_col = trg_col
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        src_text = self.df.iloc[index][self.src_col]
        trg_text = self.df.iloc[index][self.trg_col]

        # Convert text to indices
        src_indices = [self.src_vocab.stoi["<SOS>"]] + \
                      self.src_vocab.numericalize(src_text) + \
                      [self.src_vocab.stoi["<EOS>"]]

        trg_indices = [self.trg_vocab.stoi["<SOS>"]] + \
                      self.trg_vocab.numericalize(trg_text) + \
                      [self.trg_vocab.stoi["<EOS>"]]

        return torch.tensor(src_indices), torch.tensor(trg_indices)

# Collate function to pad batches to the same length
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        src = [item[0] for item in batch]
        trg = [item[1] for item in batch]

        # Pad sequences
        src = torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=self.pad_idx)
        trg = torch.nn.utils.rnn.pad_sequence(trg, batch_first=True, padding_value=self.pad_idx)

        return src, trg

In [ ]:
def load_pretrained_embeddings(vocab, file_path, emb_dim=300):
    """
    Args:
        vocab: Your Vocabulary object (src_vocab or trg_vocab)
        file_path: Path to the .vec file (e.g., 'wiki.id.vec')
        emb_dim: Dimension of the vectors (FastText is usually 300)
    Returns:
        FloatTensor of shape (vocab_size, emb_dim)
    """
    print(f"Loading embeddings from {file_path}...")

    vocab_size = len(vocab)
    embedding_matrix = torch.randn(vocab_size, emb_dim)

    hit_count = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        next(f)

        for line in tqdm(f, desc="Parsing Vectors"):
            parts = line.split()
            word = parts[0]

            if word in vocab.stoi:
                idx = vocab.stoi[word]
                try:
                    vector = torch.tensor([float(x) for x in parts[1:]])
                    if vector.shape[0] == emb_dim:
                        embedding_matrix[idx] = vector
                        hit_count += 1
                except:
                    continue

    print(f"Found {hit_count} / {vocab_size} words in pretrained file.")
    print(f"Coverage: {hit_count / vocab_size * 100:.2f}%")

    return embedding_matrix

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = True

        # 1. Bidirectional
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers,
                            dropout=dropout, batch_first=True,
                            bidirectional=True)

        self.dropout = nn.Dropout(dropout)

        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src = [batch size, src len]
        embedded = self.dropout(self.embedding(src))

        # outputs = [batch size, src len, hid_dim * 2]
        # hidden = [n_layers * 2, batch size, hid_dim]
        # cell = [n_layers * 2, batch size, hid_dim]
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Handle the Hidden/Cell States
        # We need to concatenate the forward and backward states of the LAST layer
        # hidden[-2, :, :] is the Forward state of the last layer
        # hidden[-1, :, :] is the Backward state of the last layer

        # Concatenate forward + backward
        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        cell_cat = torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)

        # Squash them through the linear layer to match Decoder size
        # shape becomes [batch size, hid_dim]
        hidden = torch.tanh(self.fc_hidden(hidden_cat))
        cell = torch.tanh(self.fc_cell(cell_cat))

        # 4. Prepare for Decoder
        # The decoder expects shape [n_layers, batch, hid_dim].
        # Since we squashed it to [batch, hid_dim], we typically unsqueeze
        # to fit n_layers=1 for the decoder.
        # (Note: This simple implementation assumes Decoder n_layers=1)
        return hidden.unsqueeze(0), cell.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = False

        # Decoder remains UNIDIRECTIONAL
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)

        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        # output = [batch size, 1, hid_dim]
        # hidden = [n_layers, batch, hid_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Encoder
        hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

In [ ]:
def predict_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=50):
    model.eval()

    # Numericalize
    if isinstance(sentence, str):
        tokens = src_vocab.numericalize(sentence)
    else:
        return ""

    tokens = [src_vocab.stoi["<SOS>"]] + tokens + [src_vocab.stoi["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device) # [1, seq_len]

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indexes = [trg_vocab.stoi["<SOS>"]]

    # Decode step by step
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab.stoi["<EOS>"]:
            break

    # Convert back to words
    trg_tokens = [trg_vocab.itos[i] for i in trg_indexes]

    # Remove SOS and EOS for cleaner output
    return " ".join(trg_tokens[1:-1])

In [ ]:
import pandas as pd

src_vocab = Vocabulary()
trg_vocab = Vocabulary()

print("Building vocabulary from Training data...")
train_df = pd.read_csv("https://drive.google.com/uc?id=12l_YUBjcAT3CnEYCqmMQwFwTj9ft5N7M")
src_vocab.build_vocabulary(train_df['text'].tolist())
trg_vocab.build_vocabulary(train_df['english_translation'].tolist())

print(f"Source Vocab Size: {len(src_vocab)}")
print(f"Target Vocab Size: {len(trg_vocab)}")

Building vocabulary from Training data...
Source Vocab Size: 8697
Target Vocab Size: 9394


In [ ]:
ENC_EMB_DIM = 300
DEC_EMB_DIM = 300
HID_DIM = 256
N_LAYERS = 1
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

In [ ]:
# pt_info = MODEL_INFO["custom_pt"]
# pt_path = os.path.join(DOWNLOAD_DIR, pt_info['filename'])

# # Download the .pt file
# print(f"Downloading {pt_info['filename']}...")
# gdown.download(id=pt_info['id'], output=pt_path, quiet=True)

# 2. Instantiate Model
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(len(src_vocab), ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
decoder = Decoder(len(trg_vocab), DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

# 2. Load the .pt file
MODEL_PATH = '/content/drive/MyDrive/Model/LSTM2.pt'
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)

# 3. Apply the weights
# This loads the weights (state_dict) into the model's architecture
model.load_state_dict(state_dict)
model.eval() # Set model to evaluation mode

print("Model successfully loaded and ready for prediction.")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


Model successfully loaded and ready for prediction.


## Inference

In [ ]:
# text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    translation = predict_sentence(
        model,
        text_to_translate,
        src_vocab,
        trg_vocab,
        DEVICE,
        max_len=50 # Use a sensible max_len to prevent infinite loops
    )

    # 3. Print the result
    print(f"Original Text: {text_to_translate}")
    print(f"Translation: {translation}")

Original Text: kalau mau datang kesini mending musim kemarau biar dapat air yang hijau jernih kalau hujan keruh airnya coklat
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: tempat wisata yang menarik untuk dikunjungi oleh peminat yang alami sejuk dan nyaman
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: recommended pisan ini kita saja sudah berpa kali ke sini enggak ada bosennya kumplit mencari makan rupa aneka pilihan main anak anak asa jalan jalan keliling doang apalagi mau main air banyak asik fresh lah kalo pulang
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: masyaallah bagus khusus yang kuat naik turun tangga hati-hati ada monyet
Translation: the place is is the the the the the the the the the the the the the the the the the the 

# Opus-mt-id-en

In [ ]:
# loaded_tokenizer = loaded_hf_models['helsinki_ft']['tokenizer']
# loaded_model = loaded_hf_models['helsinki_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "/content/drive/MyDrive/Model/fine_tuned_helsinki_tourism2"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

Model and tokenizer successfully loaded from /content/drive/MyDrive/Model/fine_tuned_helsinki_tourism2


In [ ]:
# Sample text for translation
text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    input_ids = loaded_tokenizer(
        text_to_translate,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids.to(loaded_model.device)

    outputs = loaded_model.generate(input_ids)

    translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Original Text: {text_to_translate}")
    print(f"Translation: {translation}")

Original Text: Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?
Translation: Is there a restaurant that serves halal food near this hotel?


# NLLB

In [ ]:
# loaded_tokenizer = loaded_hf_models['nllb_ft']['tokenizer']
# loaded_model = loaded_hf_models['nllb_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "/content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

Model and tokenizer successfully loaded from /content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2


In [ ]:
text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    TARGET_LANG_CODE = "eng_Latn"

    input_ids = loaded_tokenizer(
        text_to_translate,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids.to(loaded_model.device)

    outputs = loaded_model.generate(
        input_ids,
        forced_bos_token_id = loaded_tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
    )

    translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Original Text: {text_to_translate}")
    print(f"Target Language Code: {TARGET_LANG_CODE}")
    print(f"Translation: {translation}")

Original Text: kalau mau datang kesini mending musim kemarau biar dapat air yang hijau jernih kalau hujan keruh airnya coklat
Target Language Code: eng_Latn
Translation: ?If you want to come here, it's better during the dry season to get clear green water. If it rains, the water will be brown.
Original Text: tempat wisata yang menarik untuk dikunjungi oleh peminat yang alami sejuk dan nyaman
Target Language Code: eng_Latn
Translation: ?An interesting tourist spot to visit by nature lovers, cool and comfortable.
Original Text: recommended pisan ini kita saja sudah berpa kali ke sini enggak ada bosennya kumplit mencari makan rupa aneka pilihan main anak anak asa jalan jalan keliling doang apalagi mau main air banyak asik fresh lah kalo pulang
Target Language Code: eng_Latn
Translation: ? Recommended. We've been here several times. There's no excuse. I'm looking for food. There's a variety of choices. Kids don't want to walk around, especially if they want to play in the water. It's fun a

# mBART

In [ ]:
# loaded_tokenizer = loaded_hf_models['mbart_ft_ft']['tokenizer']
# loaded_model = loaded_hf_models['mbart_ft_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "/content/drive/MyDrive/Model/mbart_fine_tuned_essential2"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

Model and tokenizer successfully loaded from /content/drive/MyDrive/Model/mbart_fine_tuned_essential2


In [ ]:
text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    TARGET_LANG_CODE = "en_XX"

    input_ids = loaded_tokenizer(
        text_to_translate,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids.to(loaded_model.device)

    outputs = loaded_model.generate(
        input_ids,
        forced_bos_token_id=loaded_tokenizer.lang_code_to_id[TARGET_LANG_CODE]
    )

    translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Original Text: {text_to_translate}")
    print(f"Target Language Code: {TARGET_LANG_CODE}")
    print(f"Translation: {translation}")

Original Text: Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?
Target Language Code: en_XX
Translation: Is there a restaurant serving halal food near this hotel?
